In [ ]:
!pip install torch
!pip install scipy numpy pandas fastdtw

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.4/133.4 kB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for fastdtw: filename=fastdtw-0.3.4-cp312-cp312-linux_x86_64.whl size=567859 sha256=f102fb2142f55f6135f8e28625ec5a2d5cb05de27500f05d32fa1b265f7cad9a
  Stored in directory: /root/.cache/pip/wheels/ab/d0/26/b82cb0f49ae73e5e6bba4e8462fff2c9851d7bd2ec64f8891e
Successfully built fastdtw


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install tslearn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.7/401.7 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 71.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.9/59.9 MB 13.7 MB/s eta 0:00:00
  Attempting uninstall: llvmlite
    Found existing installation: llvmlite 0.43.0
    Uninstalling llvmlite-0.43.0:
      Successfully uninstalled llvmlite-0.43.0
  Attempting uninstall: numba
    Found existing installation: numba 0.60.0
    Uninstalling numba-0.60.0:
      Successfully uninstalled numba-0.60.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.66.0 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.66.0 which is incompatible.
pytensor 2.38.3 requires numba<=0.65.1,>=0.58, but you have numba 0.66.0 which 

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import scipy.io as sio
import pandas as pd
from pathlib import Path
from scipy.interpolate import interp1d
from torch.utils.data import Dataset, DataLoader
from fastdtw import fastdtw
from scipy.spatial.distance import euclidean
import random
import json
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\n DEVICE: {device}")
DATA_DIR = Path("/content/drive/MyDrive/Data")
TRAIN_DIR = DATA_DIR / "Training_data"
DEV_DIR = DATA_DIR / "Dev_data"
TEST_DIR = DATA_DIR / "Test_data"
SAVE_DIR = Path("/content/drive/MyDrive/huber_models")
SAVE_DIR.mkdir(exist_ok=True)
from tslearn.metrics import SoftDTW
N_POINTS = 100
EPOCHS = 100
BATCH_SIZE = 32
RESIDUAL_SCALES = {
'Contour1': 0.05,
'Contour2': 0.03
}
HYPERPARAMS = {
    'Contour1': {
        'lr': 5e-5,
        'weight_decay': 1e-5,
        'smooth_weight': 0.01
    },
    'Contour2': {
        'lr': 1e-5,
        'weight_decay': 1e-6,
        'smooth_weight': 0.02
    },
    'Contour3': {
        'lr': 2.5e-5,
        'weight_decay': 1e-5,
        'smooth_weight': 0.01
    }
}
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


 DEVICE: cuda


In [ ]:
def resample(pts,n=100):
  try:
    pts = np.array(pts,dtype=np.float32)
    if pts.ndim != 2:
      return None
    if pts.shape[0] == 2:
      pts = pts.T
    if pts.shape[1] != 2:
      return None
    if len(pts) < 2:
      return None
    diffs = np.diff(pts,axis=0)
    arc = np.concatenate([[0],np.cumsum(np.sqrt((diffs**2).sum(1)))])
    if arc[-1]==0:
      return None
    arc /=arc[-1]
    _,idx = np.unique(arc,return_index=True)
    arc = arc[idx]
    pts = pts[idx]
    if len(arc) < 2:
      return None
    t = np.linspace(0, 1, n)
    fx = interp1d(arc,pts[:,0],kind='linear',fill_value='extrapolate')
    fy = interp1d(arc,pts[:,1],kind='linear',fill_value='extrapolate')
    pts = np.stack([fx(t),fy(t)], axis=1)
    return pts.astype(np.float32)
  except:
    return None
def normalize_triplet(prev_pts,future_pts,gt_pts):
    all_pts = np.concatenate([
        prev_pts,
        future_pts,

    ], axis=0)

    mean = all_pts.mean(axis=0)
    std = all_pts.std(axis=0) + 1e-6
    prev_pts = (prev_pts - mean) / std
    future_pts = (future_pts - mean) / std
    gt_pts = (gt_pts - mean) / std
    return prev_pts,future_pts,gt_pts,mean,std
def augment_contour(pts):
  pts=pts.copy()
  angle = np.random.uniform(-5, 5) * np.pi / 180
  c = np.cos(angle)
  s = np.sin(angle)
  rot = np.array([
        [c, -s],
        [s, c]
    ])
  pts = pts@rot.T
  pts += np.random.uniform(-0.02,0.02,size=2)
  pts *= np.random.uniform(0.98,1.02)
  return pts
def load_contour(mat):
  ann = mat['annotation']
  c1,c2,c3=[],[],[]
  for i in range(len(ann)):
    c1.append(ann[i].contour1)
    c2.append(ann[i].contour2)
    c3.append(ann[i].contour3)
  return c1,c2,c3
def build_dataset(data_dir):
   sample_c1 = []
   sample_c2 = []
   sample_c3 = []
   files = sorted(list(Path(data_dir).glob("*.mat")))
   print(f"\n Found {len(files)} files")
   for j in range(len(files)):
     try:
        mat = sio.loadmat(files[j],squeeze_me = True,struct_as_record=False)
        c1,c2,c3 = load_contour(mat)
        total_frames = len(c1)
        for i in range(total_frames):
           max_length = min(i,total_frames-i-1)
           if max_length<=0:
            continue
           for n in range(1,max_length+1):
               contour_set =  [
                   (c1,sample_c1),
                   (c2,sample_c2),
                   (c3,sample_c3),
               ]
               for contour_data,samples in contour_set:
                  prev_pts = resample(
                        contour_data[i - n],
                        N_POINTS
                  )
                  future_pts = resample(
                        contour_data[i + n],
                        N_POINTS
                  )
                  gt_pts = resample(
                        contour_data[i],
                        N_POINTS
                  )
                  if prev_pts is None:
                        continue
                  if future_pts is None:
                        continue
                  if gt_pts is None:
                        continue
                  prev_pts,future_pts,gt_pts,mean,std = normalize_triplet(prev_pts,future_pts,gt_pts)
                  linear = (prev_pts + future_pts)/2.0
                  residual = gt_pts - linear
                  samples.append({'prev': prev_pts,
                        'future': future_pts,
                        'linear': linear,
                        'residual': residual,
                        'mean':mean,
                        'std':std
                       })
     except Exception as e:
       print(f"Error: {files[j].name}")
       print(e)
   return (
        sample_c1,
        sample_c2,
        sample_c3
    )
class ResidualDataset(Dataset):
  def __init__(self,samples,augment=False):
    self.samples = samples
    self.augment = augment
  def __len__(self):
    return len(self.samples)
  def __getitem__(self,idx):
    sample = self.samples[idx]
    prev_pts = sample['prev'].copy()
    future_pts = sample['future'].copy()
    linear = sample['linear'].copy()
    residual = sample['residual'].copy()
    mean = sample['mean'].copy()
    std = sample['std'].copy()
    if self.augment:
       prev_pts = augment_contour(prev_pts)
       future_pts = augment_contour(future_pts)
    return (torch.tensor(
                prev_pts,
                dtype=torch.float32
            ),
            torch.tensor(
                future_pts,
                dtype=torch.float32
            ),
            torch.tensor(
                linear,
                dtype=torch.float32
            ),
            torch.tensor(
                residual,
                dtype=torch.float32
            ),
            torch.tensor(
                mean,
                dtype=torch.float32
            ),
            torch.tensor(
                std,
                dtype=torch.float32
            ))

In [ ]:
class ContourModel1(nn.Module):
  def __init__(self):
    super().__init__()
    self.network = nn.Sequential(
        nn.Linear(400,512),
        nn.BatchNorm1d(512),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(512, 320),
        nn.BatchNorm1d(320),
        nn.ReLU(),
        nn.Dropout(0.15),
        nn.Linear(320, 256),
        nn.BatchNorm1d(256),
        nn.ReLU(),
        nn.Dropout(0.15),
        nn.Linear(256, 192),

        nn.ReLU(),
        nn.Dropout(0.15),
        nn.Linear(192, 160),
                nn.ReLU(),
        nn.Dropout(0.1),
        nn.Linear(192, 160),
        nn.ReLU(),
        nn.Dropout(0.2),

        nn.Linear(160, 200),

    )
  def forward(self,prev,future):
    batch = prev.shape[0]
    x = torch.cat([
        prev.view(batch,-1),
        future.view(batch,-1)
    ],dim=1)
    residual = self.network(x)
    return residual.view(batch,N_POINTS,2)

In [ ]:
class ResidualContourModel(nn.Module):
  def __init__(self):
    super().__init__()
    pool_size = 1
    self.cnn_encoder = nn.Sequential(
       nn.Conv1d(2, 8, kernel_size=3, padding=1),
       nn.BatchNorm1d(8),
       nn.ReLU(),
       nn.Conv1d(8, 16, kernel_size=3, padding=1),
       nn.BatchNorm1d(16),
       nn.ReLU(),
    )
    self.adaptive_pool = nn.AdaptiveAvgPool1d(1)
    self.mlp_decoder = nn.Sequential(

            nn.Linear(16,64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 200)
        )
  def forward(self,prev,future):
    batch = prev.shape[0]
    x = torch.cat([
        prev,
        future
    ],dim=1)
    transposed_x = x.permute(0,2,1)
    encode = self.cnn_encoder(transposed_x)
    adaptx = self.adaptive_pool(encode)
    xy = adaptx.view(batch,-1)
    residual = self.mlp_decoder(xy)
    return residual.view(batch,N_POINTS,2)

In [ ]:
class ContourModel2(nn.Module):
  def __init__(self):
    super().__init__()
    self.mlp_encoder = nn.Sequential(
        nn.Linear(2,128),
        nn.ReLU(),
        nn.Dropout(0.1),)
    self.pos_encoding = nn.Parameter(torch.randn(1, 200, 128))
    encoder_layer = nn.TransformerEncoderLayer(d_model=128, nhead=8,dim_feedforward=256, batch_first=True)
    self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
    self.pool_queries = nn.Parameter(torch.randn(1, 200, 128))
    self.cross_attention = nn.MultiheadAttention(
            embed_dim=128, num_heads=8, batch_first=True
        )
    self.coordinate_projection = nn.Linear(128, 2)

    self.mlp_decoder = nn.Sequential(
        nn.Linear(400,256),
        nn.ReLU(),
        nn.Dropout(0.15),

        nn.Linear(256,200)

    )
  def forward(self,prev,future):

        batch = prev.shape[0]

        x = torch.cat([

            prev,
            future

        ], dim=1)

        encode = self.mlp_encoder(x)
        encode += self.pos_encoding
        transformer_encode = self.transformer(encode)
        queries = self.pool_queries.expand(batch, -1, -1)
        attention_out, _ = self.cross_attention(
            query=queries,
            key=transformer_encode,
            value=transformer_encode
        )
        processed_coordinates = self.coordinate_projection(attention_out)
        latent = processed_coordinates.view(batch, -1)
        residual = self.mlp_decoder(latent)
        return residual.view(batch, N_POINTS, 2)

In [ ]:
!pip install pysdtw tslearn

In [ ]:
from tslearn.metrics import SoftDTWLossPyTorch
lossed=SoftDTWLossPyTorch(gamma=0.1)

In [ ]:
def train_model(train_samples,dev_samples,model_class,model_name,contour_name):
  print(f"\n Training {model_name}")
  hp = HYPERPARAMS[contour_name]
  train_loader = DataLoader(ResidualDataset(train_samples, augment=True),
        batch_size=BATCH_SIZE,
        shuffle=True)
  dev_loader = DataLoader(ResidualDataset(dev_samples),
        batch_size=BATCH_SIZE,
        shuffle=False)
  model = model_class().to(DEVICE)
  optimizer = torch.optim.Adam(
      model.parameters(),
      lr = hp['lr'],
      weight_decay = hp['weight_decay']
  )
  huber = nn.HuberLoss(delta=0.1)
  best_dev = 999999
  patience = 40          # ← Stop after 15 epochs of no improvement
  patience_counter = 0
  best_epoch = 0
  for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    for (prev,future,linear,residual,mean,std) in train_loader:
      prev = prev.to(DEVICE)
      future = future.to(DEVICE)
      linear = linear.to(DEVICE)
      residual = residual.to(DEVICE)
      pred_residual = model(prev,future)
      if contour_name in ['Contour1','Contour2']:
        pred_final = linear + pred_residual*RESIDUAL_SCALES[contour_name]
      else:
        pred_final = linear + pred_residual
      gt_final = linear + residual
      regression_loss = huber(pred_final,gt_final)
      smooth_loss = (torch.diff(pred_final,dim=1)**2).mean()
      loss = regression_loss + hp['smooth_weight']*smooth_loss
      optimizer.zero_grad()
      loss.backward()
      optimizer.step()
      train_loss += loss.item()
    avg_train = train_loss/len(train_loader)
    model.eval()
    dev_loss = 0
    with torch.no_grad():
      for (prev,future,linear,residual,mean,std) in dev_loader:
        prev = prev.to(DEVICE)
        future = future.to(DEVICE)
        linear = linear.to(DEVICE)
        residual = residual.to(DEVICE)
        pred_residual = model(prev,future)
        if contour_name in ['Contour1','Contour2']:
          pred_final = linear + pred_residual*RESIDUAL_SCALES[contour_name]
        else:
          pred_final = linear + pred_residual
        gt_final = linear + residual
        loss = huber(pred_final,gt_final)
        dev_loss += loss.item()
      avg_dev = dev_loss/len(dev_loader)
      print(f"Epoch {epoch+1:03d} | "f"Train: {avg_train:.6f} | "f"Dev: {avg_dev:.6f}")
      if avg_dev < best_dev:
        patience_counter = 0
        best_dev = avg_dev
        torch.save(model.state_dict(),SAVE_DIR/f"{model_name}.pth")
        print("Saved")
      else:
        patience_counter += 1
        if patience_counter >= patience:
          print(f"Early stopping at epoch {epoch+1}")
          break

In [ ]:
def test_model(test_samples,model_class,model_name,contour_name):
  print(f"\n Testing {model_name}")
  pred_graph = []
  gt_graph = []
  model = model_class().to(DEVICE)
  model.load_state_dict(
    torch.load(
        SAVE_DIR / f"{model_name}.pth",
        map_location=DEVICE
    )
)
  test_loader = DataLoader(ResidualDataset(test_samples, augment=False),
        batch_size=BATCH_SIZE,
        shuffle=False)
  dtw_set = []
  model.eval()
  with torch.no_grad():
    for (prev,future,linear,residual,mean,std) in test_loader:
      prev = prev.to(DEVICE)
      future = future.to(DEVICE)
      linear = linear.to(DEVICE)
      residual = residual.to(DEVICE)
      pred_residual = model(prev,future)
      if contour_name in ['Contour1','Contour2']:
          pred_final = linear + pred_residual*RESIDUAL_SCALES[contour_name]
      else:
          pred_final = linear + pred_residual
      gt_final = linear + residual
      for i in range(len(pred_final)):
        score, _ = fastdtw(
        pred_final[i].cpu().numpy()*std[i].cpu().numpy() + mean[i].cpu().numpy(),
        gt_final[i].cpu().numpy()*std[i].cpu().numpy() + mean[i].cpu().numpy(),
        dist=euclidean
        )
        dtw_set.append(score)

  return dtw_set

In [ ]:
print("\nLoading data...")
train_c1, train_c2, train_c3 = build_dataset(TRAIN_DIR)
dev_c1, dev_c2, dev_c3 = build_dataset(DEV_DIR)
test_c1, test_c2, test_c3 = build_dataset(TEST_DIR)



Loading data...

 Found 54 files

 Found 4 files

 Found 4 files


In [ ]:
print("\nTRAINING")
train_model(
    train_c3,
    dev_c3,
    ResidualContourModel,
    "contour3_huberfinal-4876pahsad",
    "Contour3"
)

print("\n HUBER TRAINING COMPLETE")


TRAINING

 Training contour3_huberfinal-4876pahsad
Epoch 001 | Train: 0.002254 | Dev: 0.000531
Saved
Epoch 002 | Train: 0.000949 | Dev: 0.000520
Saved
Epoch 003 | Train: 0.000868 | Dev: 0.000518
Saved
Epoch 004 | Train: 0.000844 | Dev: 0.000515
Saved
Epoch 005 | Train: 0.000836 | Dev: 0.000511
Saved
Epoch 006 | Train: 0.000832 | Dev: 0.000507
Saved
Epoch 007 | Train: 0.000829 | Dev: 0.000507
Epoch 008 | Train: 0.000828 | Dev: 0.000503
Saved
Epoch 009 | Train: 0.000826 | Dev: 0.000502
Saved
Epoch 010 | Train: 0.000825 | Dev: 0.000502
Saved
Epoch 011 | Train: 0.000825 | Dev: 0.000502
Saved
Epoch 012 | Train: 0.000824 | Dev: 0.000502
Epoch 013 | Train: 0.000823 | Dev: 0.000501
Saved
Epoch 014 | Train: 0.000822 | Dev: 0.000499
Saved
Epoch 015 | Train: 0.000822 | Dev: 0.000500
Epoch 016 | Train: 0.000821 | Dev: 0.000499
Epoch 017 | Train: 0.000821 | Dev: 0.000500
Epoch 018 | Train: 0.000820 | Dev: 0.000499
Epoch 019 | Train: 0.000820 | Dev: 0.000499
Epoch 020 | Train: 0.000819 | Dev: 0.000

In [ ]:
dtw3 = test_model(
    test_c3,
    ResidualContourModel,
    "contour3_huberfinal-4876pahsad",
    "Contour3"
)



 Testing contour3_huberfinal-4876pahsad


In [ ]:
for i in range(len(dtw3)):
  print(f"DTW of contour3 : {dtw3[i]}")
import pandas as pd

# Assuming you already have these lists from your code:
# dtw_1, dtw_2, dtw_3, Avg_DTW, frameno_list, temporal_offset_list

# Create a DataFrame
df = pd.DataFrame({



    'DTW_Contour3': dtw3,

})

# Save to CSV
df.to_csv('dtw_resultscontour3final4567jhgsad.csv', index=False)

print(f"✅ CSV saved with {len(df)} rows")

DTW of contour3 : 10.871870538785645
DTW of contour3 : 16.935092711126675
DTW of contour3 : 15.950849899043725
DTW of contour3 : 26.172731063378503
DTW of contour3 : 21.767888023459694
DTW of contour3 : 21.091656904862003
DTW of contour3 : 14.514472015391831
DTW of contour3 : 19.047313271850342
DTW of contour3 : 12.416895570511922
DTW of contour3 : 17.09535544854753
DTW of contour3 : 25.149606613271917
DTW of contour3 : 13.10773018409045
DTW of contour3 : 11.48456108880607
DTW of contour3 : 17.898372582899945
DTW of contour3 : 16.041239819415985
DTW of contour3 : 21.107883252474963
DTW of contour3 : 23.163408259486992
DTW of contour3 : 33.07751745488958
DTW of contour3 : 23.346940263711286
DTW of contour3 : 21.40587721230584
DTW of contour3 : 32.35137400178581
DTW of contour3 : 7.1772889950505885
DTW of contour3 : 14.42354044663144
DTW of contour3 : 8.803037579874802
DTW of contour3 : 17.908392561762096
DTW of contour3 : 8.333247551857637
DTW of contour3 : 15.772431869852534
DTW of con